# Amazon Search / LTR — Week 4 V4 Final Validation
## Final Hybrid Candidate Retrieval: BM25 + SBERT + CLIP RRF

This notebook **closes Week 4**. It does **not** train LambdaMART, LightGBM, ListNet, or any Week 5 model.

### Final Week 4 architecture
```text
Raw query
   ├── BM25 lexical retrieval ─────── Top-100
   ├── SBERT + FAISS retrieval ───── Top-100
   └── CLIP text→image + FAISS ───── Top-100
                         ↓
                 Candidate union
                         ↓
              Reciprocal Rank Fusion
                         ↓
                    Final Top-50
```

### What V4 validates
- Rebuild only the selected **BM25 + SBERT + CLIP RRF** architecture.
- Measure real per-query latency of the selected pipeline only.
- Separate:
  - query encoding time,
  - retrieval time,
  - RRF fusion time,
  - end-to-end time.
- Evaluate:
  - NDCG@10
  - MRR
  - Recall@10
  - Recall@50
  - candidate-union Recall@100
  - candidate-union Recall@300
- Check the Week 4 roadmap target:
  - multimodal Recall@50 ≥ 10% above SBERT text-only.
- Export the final Week 4 scorecard and report-ready figures.

### Upload these 3 ZIP files
1. `amazon_week1_v4_processed_data.zip`
2. `amazon_week4_v1_sbert_faiss_results.zip`
3. `amazon_week4_v2_clip_multimodal_results.zip`

In [ ]:
# ============================================================
# 0. CONFIG
# ============================================================

TOP_K = 50
CANDIDATE_DEPTH = 100
RRF_K = 60

RRF_W_BM25 = 1.00
RRF_W_SBERT = 1.00
RRF_W_CLIP = 0.35

SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CLIP_MODEL = "openai/clip-vit-base-patch32"

KNOWN_RELEVANT_THRESHOLD = 2
LATENCY_SAMPLE_SIZE = 50
RANDOM_SEED = 42

print({
    "TOP_K": TOP_K,
    "CANDIDATE_DEPTH": CANDIDATE_DEPTH,
    "RRF_K": RRF_K,
    "RRF_W_BM25": RRF_W_BM25,
    "RRF_W_SBERT": RRF_W_SBERT,
    "RRF_W_CLIP": RRF_W_CLIP,
    "LATENCY_SAMPLE_SIZE": LATENCY_SAMPLE_SIZE,
})

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

import sys, subprocess

packages = [
    "sentence-transformers>=3.0,<6",
    "transformers>=4.45,<5.0",
    "faiss-cpu>=1.8",
    "rank-bm25>=0.2.2",
    "pyarrow>=15",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed.")

In [ ]:
# ============================================================
# 2. IMPORTS + DIRECTORIES
# ============================================================

import os, re, sys, json, time, shutil, random, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import faiss
from sklearn.metrics import ndcg_score
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
INPUT_DIR = ROOT / "week4_v4_inputs"
EXTRACT_DIR = ROOT / "week4_v4_extracted"
OUTPUT_DIR = ROOT / "week4_v4_outputs"

for p in [INPUT_DIR, EXTRACT_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 3. UPLOAD 3 ZIP FILES
# ============================================================

is_colab = False
try:
    from google.colab import files
    is_colab = True
except Exception:
    files = None

if is_colab:
    print("Upload these 3 ZIPs:")
    print("1) amazon_week1_v4_processed_data.zip")
    print("2) amazon_week4_v1_sbert_faiss_results.zip")
    print("3) amazon_week4_v2_clip_multimodal_results.zip")

    uploaded = files.upload()

    for filename, content in uploaded.items():
        with open(INPUT_DIR / filename, "wb") as f:
            f.write(content)
else:
    print("Place the 3 ZIP files in:", INPUT_DIR)

zip_files = sorted(INPUT_DIR.glob("*.zip"))

if len(zip_files) < 3:
    for z in ROOT.glob("*.zip"):
        if z.parent != INPUT_DIR:
            dst = INPUT_DIR / z.name
            if not dst.exists():
                shutil.copy2(z, dst)
    zip_files = sorted(INPUT_DIR.glob("*.zip"))

print("\nZIPs found:")
for z in zip_files:
    print(" -", z.name)

if len(zip_files) < 3:
    raise FileNotFoundError("Need Week 1 + Week 4 V1 + Week 4 V2 ZIP files.")

In [ ]:
# ============================================================
# 4. EXTRACT + AUTO-DETECT REQUIRED FILES
# ============================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

for zpath in zip_files:
    out = EXTRACT_DIR / zpath.stem
    out.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(out)

    print("Extracted:", zpath.name)

def find_one(filename):
    matches = list(EXTRACT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Missing required file: {filename}")
    return matches[0]

PRODUCTS_PATH = find_one("cleaned_products_10k.parquet")
QPR_PATH = find_one("query_product_relevance.parquet")

SBERT_PRODUCT_EMB_PATH = find_one("week4_sbert_product_embeddings.npy")
SBERT_FAISS_PATH = find_one("week4_sbert_faiss.index")
TEST_QUERIES_PATH = find_one("week4_test_queries_used.csv")
V1_METRICS_PATH = find_one("week4_text_retrieval_metrics.csv")

CLIP_IMAGE_EMB_PATH = find_one("week4_clip_image_embeddings.npy")
CLIP_MAP_PATH = find_one("week4_clip_embedding_map.csv")
CLIP_FAISS_PATH = find_one("week4_clip_faiss.index")
V2_METRICS_PATH = find_one("week4_v2_final_metrics.csv")
IMAGE_COVERAGE_PATH = find_one("week4_v2_image_coverage.csv")

print("All required files detected.")

In [ ]:
# ============================================================
# 5. LOAD + ALIGN DATA
# ============================================================

products = pd.read_parquet(PRODUCTS_PATH)
qpr = pd.read_parquet(QPR_PATH)
test_queries = pd.read_csv(TEST_QUERIES_PATH)

products["product_id"] = products["product_id"].astype(str)
qpr["product_id"] = qpr["product_id"].astype(str)
qpr["query_id"] = qpr["query_id"].astype(str)
test_queries["query_id"] = test_queries["query_id"].astype(str)

products = products.drop_duplicates("product_id").reset_index(drop=True)

sbert_product_embeddings = np.load(SBERT_PRODUCT_EMB_PATH).astype("float32")
clip_image_embeddings = np.load(CLIP_IMAGE_EMB_PATH).astype("float32")
clip_map = pd.read_csv(CLIP_MAP_PATH)

if len(products) != len(sbert_product_embeddings):
    raise ValueError(
        f"Catalog rows ({len(products)}) != SBERT embedding rows "
        f"({len(sbert_product_embeddings)})."
    )

clip_map["product_id"] = clip_map["product_id"].astype(str)
clip_catalog_indices = clip_map["catalog_index"].astype(int).to_numpy()

if len(clip_map) != len(clip_image_embeddings):
    raise ValueError(
        f"CLIP map rows ({len(clip_map)}) != CLIP embedding rows "
        f"({len(clip_image_embeddings)})."
    )

product_ids = products["product_id"].tolist()
idx_to_pid = np.array(product_ids, dtype=object)
pid_to_idx = {pid:i for i,pid in enumerate(product_ids)}
title_map = (
    products.assign(title=products.get("title", "").fillna("").astype(str))
    .set_index("product_id")["title"]
    .to_dict()
)

mapped_ids = idx_to_pid[clip_catalog_indices]
if not np.all(mapped_ids == clip_map["product_id"].to_numpy()):
    raise ValueError("CLIP catalog mapping does not align with Week 1 product order.")

gold = qpr[qpr["query_id"].isin(test_queries["query_id"])].copy()

print("=" * 80)
print("WEEK 4 V4 DATA READY")
print("=" * 80)
print("Catalog products:", len(products))
print("Queries:", len(test_queries))
print("SBERT embeddings:", sbert_product_embeddings.shape)
print("CLIP image embeddings:", clip_image_embeddings.shape)
print("CLIP-covered products:", len(clip_map))
print("Image coverage:", round(len(clip_map)/len(products)*100, 2), "%")
print("Gold relevance rows:", len(gold))

In [ ]:
# ============================================================
# 6. BUILD BM25 + LOAD FAISS INDEXES
# ============================================================

def safe_col(df, col):
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)

existing_text = safe_col(products, "product_text").str.strip()

reconstructed = (
    safe_col(products, "title").str.strip() + " " +
    safe_col(products, "brand").str.strip() + " " +
    safe_col(products, "description").str.strip() + " " +
    safe_col(products, "features").str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

products["retrieval_text"] = np.where(
    existing_text.str.len() > 0,
    existing_text,
    reconstructed
)

TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[._/-][a-z0-9]+)*", re.I)

def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())

print("Building BM25...")
tokenized_corpus = [tokenize(x) for x in products["retrieval_text"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)

print("Loading saved FAISS indexes...")
sbert_index = faiss.read_index(str(SBERT_FAISS_PATH))
clip_index = faiss.read_index(str(CLIP_FAISS_PATH))

print("SBERT vectors in index:", sbert_index.ntotal)
print("CLIP vectors in index:", clip_index.ntotal)

In [ ]:
# ============================================================
# 7. LOAD QUERY ENCODERS
# ============================================================

print("Loading SBERT:", SBERT_MODEL)
sbert_model = SentenceTransformer(SBERT_MODEL, device=device)

print("Loading CLIP:", CLIP_MODEL)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(device)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
clip_model.eval()

print("Models ready.")

In [ ]:
# ============================================================
# 8. EVALUATION HELPERS
# ============================================================

relevance_lookup = defaultdict(dict)

for row in gold[["query_id","product_id","relevance_label"]].itertuples(index=False):
    relevance_lookup[row.query_id][row.product_id] = int(row.relevance_label)

known_positives = (
    gold[gold["relevance_label"] >= KNOWN_RELEVANT_THRESHOLD]
    .groupby("query_id")["product_id"]
    .apply(lambda x:set(x.astype(str)))
    .to_dict()
)

def recall_at_k(retrieved_ids, positives, k):
    positives = set(positives)

    if not positives:
        return np.nan

    return len(set(retrieved_ids[:k]) & positives) / len(positives)

def judged_metrics(qid, score_by_pid, k=10):
    rows = gold[
        gold["query_id"] == qid
    ][["product_id","relevance_label"]].copy()

    rows["score"] = rows["product_id"].map(score_by_pid).fillna(-1e9)

    if len(rows) == 0:
        return np.nan, np.nan

    labels = rows["relevance_label"].astype(float).to_numpy()
    scores = rows["score"].astype(float).to_numpy()

    ndcg = float(
        ndcg_score(
            [labels],
            [scores],
            k=min(k, len(rows))
        )
    )

    ranked = rows.sort_values("score", ascending=False)

    mrr = 0.0
    for rank, rel in enumerate(ranked["relevance_label"].tolist(), start=1):
        if rel > 0:
            mrr = 1.0 / rank
            break

    return ndcg, mrr

def top_indices(scores, k):
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k-1)[:k]
    return idx[np.argsort(-scores[idx])]

def build_rrf_scores(bm25_idx, sbert_idx, clip_catalog_idx):
    score = {}

    for rank, idx in enumerate(bm25_idx, start=1):
        idx = int(idx)
        score[idx] = score.get(idx, 0.0) + RRF_W_BM25 / (RRF_K + rank)

    for rank, idx in enumerate(sbert_idx, start=1):
        idx = int(idx)
        score[idx] = score.get(idx, 0.0) + RRF_W_SBERT / (RRF_K + rank)

    for rank, idx in enumerate(clip_catalog_idx, start=1):
        idx = int(idx)
        score[idx] = score.get(idx, 0.0) + RRF_W_CLIP / (RRF_K + rank)

    return score

In [ ]:
# ============================================================
# 9. FINAL PIPELINE — ALL 200 QUERIES
# ============================================================

all_metric_rows = []
all_top50_rows = []
candidate_rows = []

queries = test_queries["query"].fillna("").astype(str).tolist()

# Batch encodings are used here for the FINAL METRIC evaluation.
# Per-query latency is measured separately later.
print("Batch-encoding SBERT queries...")
sbert_query_embeddings = sbert_model.encode(
    queries,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Batch-encoding CLIP text queries...")
clip_query_batches = []

with torch.no_grad():
    batch_size = 64

    for start in range(0, len(queries), batch_size):
        batch = queries[start:start+batch_size]

        inp = clip_processor(
            text=batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        text_outputs = clip_model.text_model(
            input_ids=inp["input_ids"].to(device),
            attention_mask=inp["attention_mask"].to(device)
        )

        pooled = text_outputs.pooler_output
        feats = clip_model.text_projection(pooled)
        feats = feats / feats.norm(dim=-1, keepdim=True)

        clip_query_batches.append(
            feats.cpu().numpy().astype("float32")
        )

clip_query_embeddings = np.vstack(clip_query_batches)

print("Running final hybrid retrieval...")

for qi, row in enumerate(test_queries.itertuples(index=False)):
    qid, query = row.query_id, row.query

    # BM25 Top-100
    bm25_scores = np.asarray(
        bm25.get_scores(tokenize(query)),
        dtype=np.float32
    )
    bm25_top = top_indices(bm25_scores, CANDIDATE_DEPTH)

    # SBERT FAISS Top-100
    _, sbert_top = sbert_index.search(
        sbert_query_embeddings[qi:qi+1],
        CANDIDATE_DEPTH
    )
    sbert_top = sbert_top[0]

    # CLIP image FAISS Top-100
    _, clip_local_top = clip_index.search(
        clip_query_embeddings[qi:qi+1],
        min(CANDIDATE_DEPTH, clip_index.ntotal)
    )
    clip_local_top = clip_local_top[0]
    clip_catalog_top = clip_catalog_indices[clip_local_top]

    # Candidate union
    candidate_union = set(map(int, bm25_top))
    candidate_union.update(map(int, sbert_top))
    candidate_union.update(map(int, clip_catalog_top))

    # Reciprocal Rank Fusion
    rrf = build_rrf_scores(
        bm25_top,
        sbert_top,
        clip_catalog_top
    )

    ranked_union = sorted(
        candidate_union,
        key=lambda idx: rrf.get(idx, 0.0),
        reverse=True
    )

    final_idx = ranked_union[:TOP_K]
    final_pids = [str(idx_to_pid[idx]) for idx in final_idx]

    positives = known_positives.get(qid, set())

    # Candidate-union recall.
    union_pids = [str(idx_to_pid[idx]) for idx in ranked_union]

    cand_recall_100 = recall_at_k(
        union_pids,
        positives,
        min(100, len(union_pids))
    )

    cand_recall_300 = recall_at_k(
        union_pids,
        positives,
        min(300, len(union_pids))
    )

    # Judged-candidate RRF scores
    judged_pids = gold.loc[
        gold["query_id"] == qid,
        "product_id"
    ].tolist()

    score_by_pid = {}

    for pid in judged_pids:
        idx = pid_to_idx.get(pid)

        if idx is not None:
            score_by_pid[pid] = float(
                rrf.get(idx, -1e9)
            )

    ndcg, mrr = judged_metrics(
        qid,
        score_by_pid,
        k=10
    )

    all_metric_rows.append({
        "model": "BM25+SBERT+CLIP_RRF_FINAL",
        "query_id": qid,
        "query": query,
        "NDCG@10_judged": ndcg,
        "MRR_judged": mrr,
        "Recall@10_full": recall_at_k(final_pids, positives, 10),
        "Recall@50_full": recall_at_k(final_pids, positives, 50),
        "CandidateRecall@100": cand_recall_100,
        "CandidateRecall@300": cand_recall_300,
        "union_size": len(candidate_union),
    })

    for rank, idx in enumerate(final_idx, start=1):
        pid = str(idx_to_pid[idx])

        all_top50_rows.append({
            "query_id": qid,
            "query": query,
            "rank": rank,
            "product_id": pid,
            "title": title_map.get(pid, ""),
            "relevance_label": relevance_lookup[qid].get(pid, np.nan),
            "rrf_score": float(rrf.get(idx, 0.0)),
        })

    candidate_rows.append({
        "query_id": qid,
        "query": query,
        "union_size": len(candidate_union),
        "CandidateRecall@100": cand_recall_100,
        "CandidateRecall@300": cand_recall_300,
    })

    if (qi + 1) % 25 == 0 or (qi + 1) == len(test_queries):
        print(f"{qi+1}/{len(test_queries)} queries")

final_qmetrics = pd.DataFrame(all_metric_rows)
final_top50 = pd.DataFrame(all_top50_rows)
candidate_diagnostics = pd.DataFrame(candidate_rows)

print("Final retrieval evaluation complete.")

In [ ]:
# ============================================================
# 10. FINAL METRIC SCORECARD
# ============================================================

final_summary = pd.DataFrame([{
    "model": "BM25+SBERT+CLIP_RRF_FINAL",
    "NDCG@10_judged": final_qmetrics["NDCG@10_judged"].mean(),
    "MRR_judged": final_qmetrics["MRR_judged"].mean(),
    "Recall@10_full": final_qmetrics["Recall@10_full"].mean(),
    "Recall@50_full": final_qmetrics["Recall@50_full"].mean(),
    "CandidateRecall@100": final_qmetrics["CandidateRecall@100"].mean(),
    "CandidateRecall@300": final_qmetrics["CandidateRecall@300"].mean(),
    "mean_union_size": final_qmetrics["union_size"].mean(),
    "queries_evaluated": len(final_qmetrics),
}])

v1_metrics = pd.read_csv(V1_METRICS_PATH)
v2_metrics = pd.read_csv(V2_METRICS_PATH)

baseline_cols = [
    "model",
    "NDCG@10_judged",
    "MRR_judged",
    "Recall@10_full",
    "Recall@50_full",
    "queries_evaluated"
]

comparison = pd.concat([
    v1_metrics[baseline_cols],
    v2_metrics[
        v2_metrics["model"].isin(["CLIP-image", "SBERT+CLIP"])
    ][baseline_cols],
    final_summary[baseline_cols]
], ignore_index=True).drop_duplicates("model")

sbert_recall50 = comparison.loc[
    comparison["model"] == "SBERT+FAISS",
    "Recall@50_full"
].iloc[0]

target_recall50 = sbert_recall50 * 1.10

comparison["Recall@50_vs_SBERT_%"] = (
    (comparison["Recall@50_full"] - sbert_recall50)
    / sbert_recall50 * 100
)

comparison["Meets_Week4_10pct_Gate"] = (
    comparison["Recall@50_full"] >= target_recall50
)

print("SBERT Recall@50:", round(float(sbert_recall50), 4))
print("Required Week 4 target:", round(float(target_recall50), 4))
display(comparison.round(4))

print("\nFinal candidate-union diagnostics:")
display(final_summary.round(4))

In [ ]:
# ============================================================
# 11. FINAL ARCHITECTURE LATENCY BENCHMARK
# ============================================================

latency_queries = test_queries.head(
    min(LATENCY_SAMPLE_SIZE, len(test_queries))
).copy()

# Warm-up
warm_query = latency_queries.iloc[0]["query"]

_ = sbert_model.encode(
    [warm_query],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
).astype("float32")

with torch.no_grad():
    warm_inp = clip_processor(
        text=[warm_query],
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    warm_out = clip_model.text_model(
        input_ids=warm_inp["input_ids"].to(device),
        attention_mask=warm_inp["attention_mask"].to(device)
    )

    warm_feat = clip_model.text_projection(
        warm_out.pooler_output
    )
    warm_feat = warm_feat / warm_feat.norm(
        dim=-1,
        keepdim=True
    )

    warm_clip = warm_feat.cpu().numpy().astype("float32")

_ = sbert_index.search(_, CANDIDATE_DEPTH)
_ = clip_index.search(warm_clip, CANDIDATE_DEPTH)
_ = bm25.get_scores(tokenize(warm_query))

latency_records = []

for row in latency_queries.itertuples(index=False):
    qid, query = row.query_id, row.query

    # -----------------------------
    # SBERT query encoding
    # -----------------------------
    t0 = time.perf_counter()

    sbert_q = sbert_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    sbert_encode_ms = (time.perf_counter() - t0) * 1000

    # -----------------------------
    # CLIP text query encoding
    # -----------------------------
    t0 = time.perf_counter()

    with torch.no_grad():
        inp = clip_processor(
            text=[query],
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        text_outputs = clip_model.text_model(
            input_ids=inp["input_ids"].to(device),
            attention_mask=inp["attention_mask"].to(device)
        )

        pooled = text_outputs.pooler_output
        clip_feat = clip_model.text_projection(pooled)
        clip_feat = clip_feat / clip_feat.norm(
            dim=-1,
            keepdim=True
        )

        clip_q = clip_feat.cpu().numpy().astype("float32")

    clip_encode_ms = (time.perf_counter() - t0) * 1000

    # -----------------------------
    # BM25 Top-100
    # -----------------------------
    t0 = time.perf_counter()

    bm25_scores = np.asarray(
        bm25.get_scores(tokenize(query)),
        dtype=np.float32
    )

    bm25_top = top_indices(
        bm25_scores,
        CANDIDATE_DEPTH
    )

    bm25_ms = (time.perf_counter() - t0) * 1000

    # -----------------------------
    # SBERT FAISS Top-100
    # -----------------------------
    t0 = time.perf_counter()

    _, sbert_top = sbert_index.search(
        sbert_q,
        CANDIDATE_DEPTH
    )

    sbert_top = sbert_top[0]

    sbert_faiss_ms = (time.perf_counter() - t0) * 1000

    # -----------------------------
    # CLIP FAISS Top-100
    # -----------------------------
    t0 = time.perf_counter()

    _, clip_local_top = clip_index.search(
        clip_q,
        min(CANDIDATE_DEPTH, clip_index.ntotal)
    )

    clip_local_top = clip_local_top[0]
    clip_catalog_top = clip_catalog_indices[
        clip_local_top
    ]

    clip_faiss_ms = (time.perf_counter() - t0) * 1000

    # -----------------------------
    # RRF candidate union + Top-50
    # -----------------------------
    t0 = time.perf_counter()

    rrf = build_rrf_scores(
        bm25_top,
        sbert_top,
        clip_catalog_top
    )

    ranked = sorted(
        rrf,
        key=rrf.get,
        reverse=True
    )[:TOP_K]

    rrf_ms = (time.perf_counter() - t0) * 1000

    retrieval_only_ms = (
        bm25_ms +
        sbert_faiss_ms +
        clip_faiss_ms +
        rrf_ms
    )

    encoding_ms = (
        sbert_encode_ms +
        clip_encode_ms
    )

    end_to_end_ms = (
        encoding_ms +
        retrieval_only_ms
    )

    latency_records.append({
        "query_id": qid,
        "query": query,
        "SBERT_encode_ms": sbert_encode_ms,
        "CLIP_encode_ms": clip_encode_ms,
        "BM25_ms": bm25_ms,
        "SBERT_FAISS_ms": sbert_faiss_ms,
        "CLIP_FAISS_ms": clip_faiss_ms,
        "RRF_fusion_ms": rrf_ms,
        "encoding_total_ms": encoding_ms,
        "retrieval_only_ms": retrieval_only_ms,
        "end_to_end_ms": end_to_end_ms,
    })

latency_per_query = pd.DataFrame(latency_records)

def summarize_latency(col):
    a = latency_per_query[col].to_numpy(dtype=float)

    return {
        "component": col,
        "n_queries": len(a),
        "mean_ms": float(np.mean(a)),
        "median_ms": float(np.median(a)),
        "p95_ms": float(np.percentile(a, 95)),
        "max_ms": float(np.max(a)),
    }

latency_summary = pd.DataFrame([
    summarize_latency("SBERT_encode_ms"),
    summarize_latency("CLIP_encode_ms"),
    summarize_latency("BM25_ms"),
    summarize_latency("SBERT_FAISS_ms"),
    summarize_latency("CLIP_FAISS_ms"),
    summarize_latency("RRF_fusion_ms"),
    summarize_latency("encoding_total_ms"),
    summarize_latency("retrieval_only_ms"),
    summarize_latency("end_to_end_ms"),
])

display(latency_summary.round(3))

In [ ]:
# ============================================================
# 12. WEEK 4 GATES
# ============================================================

final_recall50 = float(
    final_summary["Recall@50_full"].iloc[0]
)

recall_gate_pass = (
    final_recall50 >= target_recall50
)

retrieval_p95 = float(
    latency_summary.loc[
        latency_summary["component"] == "retrieval_only_ms",
        "p95_ms"
    ].iloc[0]
)

e2e_p95 = float(
    latency_summary.loc[
        latency_summary["component"] == "end_to_end_ms",
        "p95_ms"
    ].iloc[0]
)

retrieval_latency_pass = retrieval_p95 < 50

print("=" * 85)
print("WEEK 4 FINAL VALIDATION")
print("=" * 85)

print(
    f"Multimodal Recall@50 target: "
    f"{final_recall50:.4f} vs required {target_recall50:.4f} -> "
    f'{"PASS" if recall_gate_pass else "NOT YET"}'
)

print(
    f"Retrieval-only P95 latency: "
    f"{retrieval_p95:.3f} ms -> "
    f'{"PASS" if retrieval_latency_pass else "NOT YET"}'
)

print(
    f"End-to-end P95 including SBERT + CLIP query encoding: "
    f"{e2e_p95:.3f} ms"
)

print(
    "Catalog validation size:",
    len(products),
    "products"
)

print("=" * 85)

In [ ]:
# ============================================================
# 13. PER-QUERY FINAL ANALYSIS
# ============================================================

# Reuse V1 SBERT query metrics for direct comparison.
sbert_metric_paths = list(
    EXTRACT_DIR.rglob("week4_sbert_query_metrics.csv")
)

if sbert_metric_paths:
    sbert_q = pd.read_csv(sbert_metric_paths[0])
    sbert_q["query_id"] = sbert_q["query_id"].astype(str)

    per_query_comparison = final_qmetrics.merge(
        sbert_q[[
            "query_id",
            "NDCG@10_judged",
            "Recall@50_full"
        ]],
        on="query_id",
        how="left",
        suffixes=("_FINAL", "_SBERT")
    )

    per_query_comparison[
        "Recall50_FINAL_minus_SBERT"
    ] = (
        per_query_comparison["Recall@50_full_FINAL"]
        -
        per_query_comparison["Recall@50_full_SBERT"]
    )

    per_query_comparison[
        "NDCG_FINAL_minus_SBERT"
    ] = (
        per_query_comparison["NDCG@10_judged_FINAL"]
        -
        per_query_comparison["NDCG@10_judged_SBERT"]
    )

    print("Largest Recall@50 gains:")
    display(
        per_query_comparison.sort_values(
            "Recall50_FINAL_minus_SBERT",
            ascending=False
        ).head(15)[[
            "query_id",
            "query",
            "Recall@50_full_SBERT",
            "Recall@50_full_FINAL",
            "Recall50_FINAL_minus_SBERT",
            "NDCG@10_judged_SBERT",
            "NDCG@10_judged_FINAL",
        ]]
    )

    print("\nLargest Recall@50 losses:")
    display(
        per_query_comparison.sort_values(
            "Recall50_FINAL_minus_SBERT"
        ).head(15)[[
            "query_id",
            "query",
            "Recall@50_full_SBERT",
            "Recall@50_full_FINAL",
            "Recall50_FINAL_minus_SBERT",
            "NDCG@10_judged_SBERT",
            "NDCG@10_judged_FINAL",
        ]]
    )
else:
    per_query_comparison = final_qmetrics.copy()
    print("V1 per-query SBERT file not found; skipped direct per-query comparison.")

In [ ]:
# ============================================================
# 14. REPORT-READY PLOTS
# ============================================================

plot_cols = [
    "NDCG@10_judged",
    "MRR_judged",
    "Recall@10_full",
    "Recall@50_full"
]

ax = comparison.set_index("model")[plot_cols].T.plot(
    kind="bar",
    figsize=(12, 6)
)

ax.set_title(
    "Week 4 Final — BM25 vs SBERT vs CLIP vs Hybrid RRF"
)
ax.set_ylabel("Metric")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)

plt.xticks(rotation=25, ha="right")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "week4_v4_final_metric_comparison.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


lat_plot = latency_summary[
    latency_summary["component"].isin([
        "BM25_ms",
        "SBERT_FAISS_ms",
        "CLIP_FAISS_ms",
        "RRF_fusion_ms",
        "retrieval_only_ms",
        "end_to_end_ms"
    ])
].copy()

ax = lat_plot.set_index("component")[
    ["mean_ms", "p95_ms"]
].plot(
    kind="bar",
    figsize=(11, 5)
)

ax.set_title(
    "Week 4 Final — Hybrid Retrieval Latency"
)
ax.set_ylabel("Milliseconds")
ax.grid(axis="y", alpha=0.25)

plt.xticks(rotation=25, ha="right")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "week4_v4_final_latency.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 15. EXPORT FINAL WEEK 4 ARTIFACTS
# ============================================================

final_qmetrics.to_csv(
    OUTPUT_DIR / "week4_v4_final_query_metrics.csv",
    index=False
)

final_top50.to_csv(
    OUTPUT_DIR / "week4_v4_final_top50.csv",
    index=False
)

candidate_diagnostics.to_csv(
    OUTPUT_DIR / "week4_v4_candidate_union_diagnostics.csv",
    index=False
)

final_summary.to_csv(
    OUTPUT_DIR / "week4_v4_final_summary.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR / "week4_v4_final_scorecard.csv",
    index=False
)

latency_per_query.to_csv(
    OUTPUT_DIR / "week4_v4_latency_per_query.csv",
    index=False
)

latency_summary.to_csv(
    OUTPUT_DIR / "week4_v4_latency_summary.csv",
    index=False
)

per_query_comparison.to_csv(
    OUTPUT_DIR / "week4_v4_per_query_analysis.csv",
    index=False
)

image_coverage = pd.read_csv(IMAGE_COVERAGE_PATH)
image_coverage.to_csv(
    OUTPUT_DIR / "week4_v4_image_coverage_reference.csv",
    index=False
)

gate_summary = pd.DataFrame([{
    "catalog_products": len(products),
    "test_queries": len(test_queries),
    "sbert_recall50": sbert_recall50,
    "required_multimodal_recall50": target_recall50,
    "final_hybrid_recall50": final_recall50,
    "relative_recall50_vs_sbert_pct":
        (final_recall50 - sbert_recall50) / sbert_recall50 * 100,
    "multimodal_recall_gate_pass": recall_gate_pass,
    "retrieval_only_p95_ms": retrieval_p95,
    "retrieval_only_sub50ms_pass": retrieval_latency_pass,
    "end_to_end_p95_ms": e2e_p95,
}])

gate_summary.to_csv(
    OUTPUT_DIR / "week4_v4_final_gates.csv",
    index=False
)

readme = f'''
Amazon Search / LTR — Week 4 Final Validation
==============================================

Final architecture
------------------
BM25 Top-{CANDIDATE_DEPTH}
+
SBERT FAISS Top-{CANDIDATE_DEPTH}
+
CLIP image FAISS Top-{CANDIDATE_DEPTH}
->
candidate union
->
weighted reciprocal-rank fusion
->
Top-{TOP_K}

RRF configuration
-----------------
k = {RRF_K}
BM25 weight = {RRF_W_BM25}
SBERT weight = {RRF_W_SBERT}
CLIP weight = {RRF_W_CLIP}

Week 4 gates
------------
SBERT Recall@50 = {sbert_recall50:.4f}
Required multimodal Recall@50 = {target_recall50:.4f}
Final hybrid Recall@50 = {final_recall50:.4f}
Recall gate = {"PASS" if recall_gate_pass else "NOT YET"}

Retrieval-only P95 = {retrieval_p95:.3f} ms
Sub-50ms retrieval gate = {"PASS" if retrieval_latency_pass else "NOT YET"}

End-to-end P95 including query encoders = {e2e_p95:.3f} ms

Important interpretation
------------------------
The final hybrid architecture treats BM25, SBERT, and CLIP as independent
candidate sources. CLIP is not used as a global penalty for products that
lack images. This directly addresses the partial-image-coverage failure
observed in the naive Week 4 V2 score-fusion experiment.

No Week 5 learning-to-rank model is trained in this notebook.
'''

with open(
    OUTPUT_DIR / "README_WEEK4_FINAL.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)

zip_path = shutil.make_archive(
    str(ROOT / "amazon_week4_v4_final_validation_results"),
    "zip",
    OUTPUT_DIR
)

print("Created:", zip_path)

In [ ]:
# ============================================================
# 16. FINAL DISPLAY + DOWNLOAD
# ============================================================

print("=" * 90)
print("WEEK 4 FINAL VALIDATION COMPLETE")
print("=" * 90)

display(gate_summary.round(4))
display(final_summary.round(4))
display(comparison.round(4))
display(latency_summary.round(3))

print("\nNo Week 5 model was trained.")

if is_colab:
    files.download(
        str(
            ROOT /
            "amazon_week4_v4_final_validation_results.zip"
        )
    )
else:
    print(
        ROOT /
        "amazon_week4_v4_final_validation_results.zip"
    )